# Tính chỉ số lợi nhuận, rủi ro, thanh khoản và tạo CSDL SQLite

Notebook đọc `du_lieu_sach.csv` (kết quả của notebook 01), tính các chỉ số theo từng mã, tổng hợp cho 35 mã ngành Hàng tiêu dùng thiết yếu, rồi xuất `du_lieu_chi_so.csv`, `bang_tong_hop_ma.csv` và `du_an.db` (SQLite) vào thư mục `2_DuLieuDaXuLy`.

**Cách chạy:** chạy notebook 01 trước, đặt notebook này cùng cấp với thư mục `2_DuLieuDaXuLy`, rồi chạy lần lượt từ trên xuống. Đường dẫn là đường dẫn tương đối nên chạy được trên mọi máy sau khi clone repo.

## 0. Thư viện
Import toàn bộ thư viện **một lần duy nhất** ở ô này, các ô sau không import lại. Đồng thời đặt chế độ hiển thị bảng (tối đa 100 dòng, rộng 220 ký tự).

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 220)

## 1. Cấu hình đường dẫn, tên cột và ngưỡng
Khai báo mọi thứ có thể cần chỉnh ở **một chỗ**: đường dẫn file (tương đối so với thư mục chứa notebook), tên cột trong `du_lieu_sach.csv` (nguồn đổi tên cột chỉ sửa ở đây), ngưỡng gắn cờ biên độ bất thường, bảng đổi tên cột có dấu sang không dấu cho SQL và danh sách cột chỉ số đưa vào SQL.

In [ ]:
# --- Đường dẫn ---
DIR_XULY_IN     = Path('data/processed/OUTPUT_FILE_1')
DIR_XULY     = Path('data/processed/OUTPUT_FILE_2')
FILE_SACH     = DIR_XULY_IN / 'du_lieu_sach.csv'        # đầu vào (từ notebook 01)
FILE_CHI_SO   = DIR_XULY / 'du_lieu_chi_so.csv'      # đầu ra 1: dữ liệu theo ngày + chỉ số
FILE_TONG_HOP = DIR_XULY / 'bang_tong_hop_ma.csv'    # đầu ra 2: tổng hợp theo mã ngành
FILE_DB       = DIR_XULY / 'du_an.db'                # đầu ra 3: CSDL SQLite

# --- Tên cột trong du_lieu_sach.csv ---
COL = {
    'MA': 'Mã', 'NGAY': 'Ngày', 'TEN': 'TÊN DN NIÊM YẾT',
    'GIA_DC': 'Giá điều chỉnh', 'GIA_MO': 'Giá mở cửa', 'GIA_CAO': 'Giá cao nhất', 'GIA_THAP': 'Giá thấp nhất',
    'KL_KHOP': 'Khối lượng khớp lệnh', 'GT_KHOP': 'Giá trị khớp lệnh', 'KL_LH': 'KL LƯU HÀNH',
}
M, D = COL['MA'], COL['NGAY']

# --- Ngưỡng ---
NGUONG_BIEN_DO = 0.075   # |lợi nhuận ngày| > 7,5% (7% biên độ HOSE + 0,5 điểm dung sai): chỉ gắn cờ, không xóa

# --- Cấu hình cho SQL ---
TEN_COT_SQL = {  # tên có dấu -> tên không dấu
    'Mã': 'Ma', 'Ngày': 'Ngay', 'Giá điều chỉnh': 'Gia_dc', 'Giá đóng cửa': 'Gia_dong',
    'Khối lượng khớp lệnh': 'KL_khop', 'Giá trị khớp lệnh': 'GT_khop',
    'Khối lượng thỏa thuận': 'KL_tt', 'Giá trị thỏa thuận': 'GT_tt',
}
COT_CHI_SO = ['Is_Staples', 'Nhom_du_lieu', 'Truoc_ngay_niem_yet', 'Ngay_thieu_du_lieu',
              'Daily_Return', 'Cumulative_Return', 'Intraday_Volatility', 'Is_Down_Day',
              'Turnover_Ratio', 'Drawdown', 'Max_Drawdown', 'Is_Zero_Volume',
              'Nghi_van_bien_do', 'Industry_Avg_Return']

## 2. Hàm dùng chung
`bang()` đổi dict "chỉ tiêu -> giá trị" thành DataFrame 1 cột để hiển thị gọn, dùng thay cho nhiều lệnh `print`.

In [3]:
def bang(chi_tieu, ten_cot='Giá trị'):
    return pd.Series(chi_tieu, name=ten_cot, dtype=object).to_frame().rename_axis('Chỉ tiêu')

## 3. Đọc dữ liệu sạch và kiểm tra cột bắt buộc
Đọc `du_lieu_sach.csv`, sắp xếp tăng dần theo Mã và Ngày (để các phép tính dùng "phiên trước" đúng thứ tự). Dừng lại nếu thiếu 3 cột cờ do notebook 01 tạo (`Is_Staples`, `Nhom_du_lieu`, `Truoc_ngay_niem_yet`). Bảng kết quả cho biết số dòng và số mã đã đọc.

In [4]:
df = pd.read_csv(FILE_SACH, encoding='utf-8-sig', parse_dates=[D])
df = df.sort_values([M, D]).reset_index(drop=True)

for c in ['Is_Staples', 'Nhom_du_lieu', 'Truoc_ngay_niem_yet']:
    assert c in df.columns, f'Thiếu cột {c}: cần chạy lại notebook 01'

display(bang({'Số dòng': len(df), 'Số mã': df[M].nunique()}))

,Giá trị
Chỉ tiêu,
Số dòng,195268
Số mã,419


## 4. Tính chỉ số theo từng mã và từng ngày
Mỗi chỉ số được tính riêng cho từng mã (`gia` = Giá điều chỉnh):
- `Daily_Return = gia(t) / gia(t-1) - 1`: lợi nhuận ngày.
- `Cumulative_Return = gia(t) / gia(ngày đầu) - 1`: lợi nhuận tích lũy từ phiên đầu của mã.
- `Intraday_Volatility = (Giá cao nhất - Giá thấp nhất) / Giá mở cửa`: biên độ giật giá trong phiên (bỏ trống nếu giá mở cửa = 0).
- `Is_Down_Day = 1` nếu `Daily_Return < 0`.
- `Turnover_Ratio = KL khớp lệnh / KL lưu hành`: tỷ lệ quay vòng cổ phiếu trong ngày.
- `Drawdown = gia(t) / max(gia từ đầu đến t) - 1`; `Max_Drawdown` = giá trị nhỏ nhất của `Drawdown` theo mã (mức sụt sâu nhất từ đỉnh).
- `Is_Zero_Volume = 1` nếu KL khớp lệnh = 0; `Zero_Volume_Days` = tổng số ngày như vậy theo mã.
- `Nghi_van_bien_do = 1` nếu `|Daily_Return| > 7,5%` (chỉ gắn cờ).

In [5]:
gia = df[COL['GIA_DC']]
theo_ma = df.groupby(M)

df['Daily_Return']       = gia / theo_ma[COL['GIA_DC']].shift(1) - 1
df['Cumulative_Return']  = gia / theo_ma[COL['GIA_DC']].transform('first') - 1
mo = df[COL['GIA_MO']]
df['Intraday_Volatility'] = np.where(mo > 0, (df[COL['GIA_CAO']] - df[COL['GIA_THAP']]) / mo, np.nan)
df['Is_Down_Day']        = (df['Daily_Return'] < 0).astype(int)
df['Turnover_Ratio']     = (df[COL['KL_KHOP']] / df[COL['KL_LH']]).replace([np.inf, -np.inf], np.nan)
df['Drawdown']           = gia / theo_ma[COL['GIA_DC']].cummax() - 1
df['Max_Drawdown']       = df.groupby(M)['Drawdown'].transform('min')
df['Is_Zero_Volume']     = (df[COL['KL_KHOP']] == 0).astype(int)
df['Zero_Volume_Days']   = df.groupby(M)['Is_Zero_Volume'].transform('sum')
df['Nghi_van_bien_do']   = (df['Daily_Return'].abs() > NGUONG_BIEN_DO).astype(int)

## 5. Tính mốc so sánh của cả ngành
Dùng để so từng mã với ngành:
- `Industry_Avg_Return(ngày) = trung bình Daily_Return của các mã ngành tiêu dùng (Is_Staples = 1) đang giao dịch trong ngày đó`.
- `Industry_Cum_Return(t) = tích (1 + Industry_Avg_Return) từ ngày đầu đến t - 1` (ngày chưa có lợi nhuận được coi là 0).

Hai cột này được ghép vào bảng chính theo ngày.

In [6]:
tb_nganh = df[df['Is_Staples'] == 1].groupby(D)['Daily_Return'].mean().rename('Industry_Avg_Return')
nganh_ngay = tb_nganh.to_frame()
nganh_ngay['Industry_Cum_Return'] = (1 + nganh_ngay['Industry_Avg_Return'].fillna(0)).cumprod() - 1
df = df.merge(nganh_ngay, left_on=D, right_index=True, how='left')

## 6. Bảng tổng hợp theo mã ngành
Gom dữ liệu ngày của 35 mã ngành thành một dòng mỗi mã:
- `Tong_loi_nhuan` = `Cumulative_Return` của ngày cuối; `Max_Drawdown` = mức sụt sâu nhất từ đỉnh.
- `Do_lech_chuan_ngay` = độ lệch chuẩn của `Daily_Return`; `Bien_do_phien_TB` = trung bình `Intraday_Volatility`.
- `Ty_le_ngay_giam` = trung bình `Is_Down_Day` (= số ngày giảm / số ngày); `Turnover_TB` = trung bình `Turnover_Ratio`.
- `Zero_Volume_Days`, `Tong_gia_tri_khop` (tỷ đồng, tổng giá trị khớp lệnh), `So_ngay_bien_do_bat_thuong` (tổng cờ `Nghi_van_bien_do`).
- `Loi_nhuan_tu_niem_yet = giá cuối / giá đầu - 1`, chỉ tính từ các dòng sau ngày niêm yết trên HOSE (`Truoc_ngay_niem_yet = 0`).

In [7]:
st = df[df['Is_Staples'] == 1]

tong_hop = st.groupby(M).agg(
    Nhom_du_lieu=('Nhom_du_lieu', 'first'),
    So_phien=('So_phien', 'first'),
    Tong_loi_nhuan=('Cumulative_Return', 'last'),
    Max_Drawdown=('Max_Drawdown', 'first'),
    Do_lech_chuan_ngay=('Daily_Return', 'std'),
    Bien_do_phien_TB=('Intraday_Volatility', 'mean'),
    Ty_le_ngay_giam=('Is_Down_Day', 'mean'),
    Turnover_TB=('Turnover_Ratio', 'mean'),
    Zero_Volume_Days=('Zero_Volume_Days', 'first'),
    Tong_gia_tri_khop=(COL['GT_KHOP'], 'sum'),
    So_ngay_bien_do_bat_thuong=('Nghi_van_bien_do', 'sum'),
).reset_index()

sau = st[st['Truoc_ngay_niem_yet'] == 0].groupby(M)[COL['GIA_DC']].agg(['first', 'last'])
tong_hop['Loi_nhuan_tu_niem_yet'] = tong_hop[M].map(sau['last'] / sau['first'] - 1)

## 7. Xuất file
Đổi Ngày sang dạng `yyyy-mm-dd` (trên bản sao, không đổi `df`) để Power BI/SQL đọc đúng kiểu ngày, rồi ghi `du_lieu_chi_so.csv` (dữ liệu ngày + chỉ số) và `bang_tong_hop_ma.csv` (tổng hợp theo mã ngành).

In [8]:
df_xuat = df.assign(**{D: df[D].dt.strftime('%Y-%m-%d')})
df_xuat.to_csv(FILE_CHI_SO, index=False, encoding='utf-8-sig')
tong_hop.to_csv(FILE_TONG_HOP, index=False, encoding='utf-8-sig')

## 8. Kiểm tra kết quả
(1) Số giá trị vô cực (`inf`) trong các cột chỉ số, phải bằng 0 và số dòng bảng tổng hợp phải bằng 35.
(2) Kiểm tra bằng tay: 4 ngày đầu của 1 mã ngành đủ dữ liệu, đối chiếu `Daily_Return = gia(t)/gia(t-1) - 1` và `Cumulative_Return = gia(t)/gia(ngày 1) - 1` với giá điều chỉnh.
(3) Bảng tổng hợp ngành xếp theo lợi nhuận giảm dần.

In [9]:
cot_chi_so = ['Daily_Return', 'Cumulative_Return', 'Intraday_Volatility', 'Turnover_Ratio', 'Drawdown', 'Industry_Avg_Return']
display(bang({
    'Giá trị vô cực (inf)': int(np.isinf(df[cot_chi_so]).sum().sum()),
    'Số dòng bảng tổng hợp': len(tong_hop),
}))

ma_thu = tong_hop.loc[tong_hop['Nhom_du_lieu'] == 'Day_du', M].iloc[0]
display(df_xuat.loc[df_xuat[M] == ma_thu, [M, D, COL['GIA_DC'], 'Daily_Return', 'Cumulative_Return']].head(4))

display(tong_hop.sort_values('Tong_loi_nhuan', ascending=False).round(4))

,Giá trị
Chỉ tiêu,
Giá trị vô cực (inf),0
Số dòng bảng tổng hợp,35


,Mã,Ngày,Giá điều chỉnh,Daily_Return,Cumulative_Return
492,AAM,2024-10-21,6.63,NaN,0.000000
493,AAM,2024-10-22,6.71,0.012066,0.012066
494,AAM,2024-10-23,6.71,0.000000,0.012066
495,AAM,2024-10-24,6.38,-0.049180,-0.037707


,Mã,Nhom_du_lieu,So_phien,Tong_loi_nhuan,Max_Drawdown,Do_lech_chuan_ngay,Bien_do_phien_TB,Ty_le_ngay_giam,Turnover_TB,Zero_Volume_Days,Tong_gia_tri_khop,So_ngay_bien_do_bat_thuong,Loi_nhuan_tu_niem_yet
25,NAF,Day_du,492,1.4218,-0.1791,0.0162,0.0220,0.3984,0.0083,0,8835.33,0,1.4218
29,SBT,Day_du,492,1.1388,-0.2344,0.0131,0.0182,0.3232,0.0013,0,11067.51,0,1.1388
16,HSL,Day_du,492,0.7619,-0.7044,0.0321,0.0418,0.4573,0.0110,0,1777.10,0,0.7619
7,BAF,Day_du,492,0.6428,-0.1933,0.0143,0.0207,0.4553,0.0080,0,46813.03,0,0.6428
32,VCF,Day_du,491,0.5444,-0.1910,0.0210,0.0139,0.3666,0.0000,55,124.65,0,0.5444
2,ABT,Day_du,492,0.4109,-0.2982,0.0175,0.0191,0.4065,0.0008,12,247.64,0,0.4109
9,CLC,Day_du,492,0.3695,-0.1486,0.0168,0.0156,0.3374,0.0001,76,93.44,0,0.3695
14,HAG,Day_du,492,0.3585,-0.2785,0.0218,0.0296,0.3902,0.0063,0,59348.41,0,0.3585
30,SMB,Day_du,492,0.2843,-0.1097,0.0098,0.0136,0.3638,0.0005,5,309.72,0,0.2843
22,MCH,Niem_yet_moi,481,0.2760,-0.4149,0.0219,0.0267,0.4657,0.0002,0,21884.08,7,-0.1860


## 9. Chuẩn bị bảng cho SQL
Đọc lại từ 2 file CSV đã lưu (để không phụ thuộc trạng thái các ô trước). Lấy các cột gốc + cột chỉ số, đổi tên có dấu sang không dấu (`TEN_COT_SQL`) thành bảng `gia_ngay`; thêm tên doanh nghiệp (`Ten_DN`) vào bảng tổng hợp `tong_hop_ma` bằng cách tra theo mã.

In [10]:
dl = pd.read_csv(FILE_CHI_SO, encoding='utf-8-sig')
th = pd.read_csv(FILE_TONG_HOP, encoding='utf-8-sig')

gia_ngay = dl[list(TEN_COT_SQL) + COT_CHI_SO].rename(columns=TEN_COT_SQL)

ten_dn = dl.drop_duplicates(M).set_index(M)[COL['TEN']]
th = th.rename(columns={M: 'Ma'})
th['Ten_DN'] = th['Ma'].map(ten_dn)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_21992\1528314251.py:1: DtypeWarning: Columns (0: TÊN DN NIÊM YẾT, 1: NGÀY NIÊM YẾT) have mixed types. Specify dtype option on import or set low_memory=False.
  dl = pd.read_csv(FILE_CHI_SO, encoding='utf-8-sig')


## 10. Ghi vào SQLite và kiểm tra
Tạo lại file `du_an.db` (xóa bản cũ nếu có), ghi 2 bảng `gia_ngay`, `tong_hop_ma` và tạo chỉ mục `(Ma, Ngay)` để truy vấn theo mã và ngày nhanh hơn. Bảng kết quả đếm số dòng của mỗi bảng và số mã trong `gia_ngay`.

In [11]:
FILE_DB.unlink(missing_ok=True)
con = sqlite3.connect(FILE_DB)
gia_ngay.to_sql('gia_ngay', con, index=False)
th.to_sql('tong_hop_ma', con, index=False)
con.execute('CREATE INDEX ix_gia_ma_ngay ON gia_ngay(Ma, Ngay)')
con.commit()

so_dong, so_ma = con.execute('SELECT COUNT(*), COUNT(DISTINCT Ma) FROM gia_ngay').fetchone()
so_th = con.execute('SELECT COUNT(*) FROM tong_hop_ma').fetchone()[0]
con.close()

display(bang({'Số dòng gia_ngay': so_dong, 'Số mã trong gia_ngay': so_ma,
              'Số dòng tong_hop_ma': so_th, 'File đã lưu': str(FILE_DB)}))

,Giá trị
Chỉ tiêu,
Số dòng gia_ngay,195268
Số mã trong gia_ngay,419
Số dòng tong_hop_ma,35
File đã lưu,2_Dulieudaxuly\du_an.db
